# Tokenizers for Guarani

This notebook introduces an experiment that explores how different state-of-the-art `tokenizers` work with Guarani text. The analysis is based on two metrics introduced by [Rust et al.](https://arxiv.org/abs/2012.15613), which examine: i) the average number of subword produced per tokenized word (`fertility`); and ii) the `proportion` of words where tokenized words are continued by at least two sub-tokens (denoted by continuation symbols, e.g., " ", "_"). For both metrics, low scores are preferred, indicating better suitability of tokenizers for the language.

Guarani text include in [FineWeb-2](https://huggingface.co/datasets/HuggingFaceFW/fineweb-2) is used for the experiment and word counter is conducted with the Spacy generic segmentator. The tokenizers included in the experiment belong to the multilingual models: [Gemma v3](https://huggingface.co/google/gemma-3-1b-it), [Qwen v3](https://huggingface.co/docs/transformers/en/model_doc/qwen3), [GPT-4o](https://huggingface.co/Xenova/gpt-4o), [Llama v3.1](https://huggingface.co/meta-llama/Llama-3.1-8B), [Phi v4](https://huggingface.co/microsoft/phi-4), [Aya v101](https://huggingface.co/microsoft/phi-4), and [MADLAD-400-3B-MT](https://huggingface.co/google/madlad400-3b-mt).

The ultimately goal of the experiment is to find the best tokenizer for Guarani text, in terms of `fertility` and `proportion of continued words`.

## Mount drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load libraries

In [2]:
import json
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import spacy

from google.colab import userdata
from transformers import AutoTokenizer
from tqdm import tqdm

## Load data

In [3]:
data_dir = '/content/drive/MyDrive/Professional/GuaranIA/2. Corpus/Raw data/fineweb2'
project_dir = '/content/drive/MyDrive/Professional/GuaranIA/2. Corpus/Experiments'

In [4]:
data_df = pd.read_csv(os.path.join(data_dir, 'fineweb2_gug_latn_all.csv'))

In [5]:
data_df.head()

,text,id,dump,url,date,file_path,language,language_score,language_script,minhash_cluster_size,top_langs,num_words_no_punct_spacy,num_words_punct_spacy,num_words_split,domain,source,filter_reason,wordlist_ratio
0,Mburuvicha guasu Lugo oñembyaty kuehe ministro...,<urn:uuid:53a9a54e-be00-41d9-8e4e-e749c6da6f40>,CC-MAIN-2013-20,http://archivo.abc.com.py/seccion.php?sec=11&f...,2013-05-23T16:54:42Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,gug,0.999867,Latn,1,{},943,1130,881,archivo.abc.com.py,fineweb2_train,NaN,NaN
1,En esta parte hemos reunido las expresiones po...,<urn:uuid:0f0c663a-a218-4539-af22-2b5309abc26c>,CC-MAIN-2013-20,http://www.datamex.com.py/guarani/neenga/neeng...,2013-05-23T17:08:05Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,gug,0.587419,Latn,11,{},1830,2124,1660,www.datamex.com.py,fineweb2_train,NaN,NaN
2,JATA’Y: LA LEYENDA\nOhai: David Galeano Oliver...,<urn:uuid:c1141018-0b4c-48a3-9ab3-6ebf6b2721fa>,CC-MAIN-2013-20,http://lengua-guarani.blog.com.es/2011/04/24/j...,2013-06-19T18:11:15Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,gug,0.999284,Latn,57,{},1033,1257,1022,lengua-guarani.blog.com.es,fineweb2_train,NaN,NaN
3,Emboscada\n|Emboscada\n\n\nEmboscada\n|Tetã||P...,<urn:uuid:61f1e0c7-910b-4cab-9dc7-e9b88c3d8a51>,CC-MAIN-2013-20,http://gn.wikipedia.org/wiki/Emboscada,2013-05-23T09:30:32Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,gug,0.999914,Latn,7,{},458,569,403,gn.wikipedia.org,fineweb2_train,NaN,NaN
4,"Universidad Americana\nUniversidad Americana, ...",<urn:uuid:efd57a22-4fcf-4788-ad1e-ccfcce2a09a8>,CC-MAIN-2013-20,http://gn.wikipedia.org/wiki/Universidad_Ameri...,2013-05-18T22:01:15Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,gug,0.999907,Latn,44,{},126,154,123,gn.wikipedia.org,fineweb2_train,NaN,NaN


## Experiment with tokenizers

In [6]:
# define operative constants
OVERWRITE_EXP = True

In [7]:
# load huggingface access token
hf_access_token = userdata.get('HF_TOKEN')

In [8]:
word_seg = spacy.blank("xx")
def text_segmentator(text, include_punct=False):
    words = []
    for t in word_seg(text):
      if not include_punct and t.is_punct:
        continue
      words.append(t.text)
    return words

In [9]:
def show_tokens(tokenizer, token_ids, sample=-1):
  color_list = ['102;194;165', '252;141;98', '141;160;203', '231;138;195', '166;216;84', '255;217;47']
  for idx, token in enumerate(token_ids):
    if sample != -1 and idx > sample:
      break
    print(
        f"\x1b[0;30;48;2;{color_list[idx % len(color_list)]}m" + tokenizer.decode(token) + "\x1b[0m",
        end=' '
    )
  print('\n')

In [10]:
def print_tokens(tokenizer, token_ids, sample=-1):
  p_tokens = []
  for idx, token in enumerate(token_ids):
    if sample != -1 and idx > sample:
      break
    p_tokens.append(tokenizer.decode(token))
  print(p_tokens)

In [11]:
def compute_fertility(num_words, num_tokens):
  return num_tokens/num_words

In [12]:
def do_compute_continued_words_madlad(token_ids, tokenizer, special_tokens, text):
  continued_words = 0
  tokens = token_ids
  words = text_segmentator(text, True)
  idx_token = 0
  for word in words:
    subwords = []
    current_word = ''
    while idx_token < len(tokens) and len(current_word) < len(word):
      decoded_token = tokenizer.decode(tokens[idx_token])
      if decoded_token not in special_tokens:
        current_word += decoded_token
        subwords.append(decoded_token)
      idx_token += 1
    if len(subwords) > 1:
      continued_words += 1
  return continued_words

In [13]:
def do_compute_continued_words(tokens, tokenizer, special_tokens,):
  tokenizer_id = tokenizer.name_or_path
  new_word = True
  continued_words = 0
  for token in tokens:
    if token in special_tokens:
      continue
    decoded_token = tokenizer.decode(token)
    if tokenizer_id == 'CohereForAI/aya-101':
      tokenizer_cond = decoded_token != ''
    else:
      tokenizer_cond = not decoded_token.startswith(' ')
    if decoded_token and tokenizer_cond and new_word:
      continued_words += 1
      new_word = False
    else:
      new_word = True
  return continued_words

In [14]:
def compute_continued_words(token_ids, tokenizer, special_tokens, text):
  tokenizer_id = tokenizer.name_or_path

  if tokenizer_id == 'google/madlad400-3b-mt':
    continued_words = do_compute_continued_words_madlad(token_ids, tokenizer, special_tokens, text)
  else:
    continued_words = do_compute_continued_words(token_ids, tokenizer, special_tokens)

  return continued_words

In [15]:
def process_text(text, tokenizer, verbose):
  # Tokenize text
  token_ids = tokenizer(text).input_ids
  if verbose: print_tokens(tokenizer, token_ids, sample=10)
  num_tokens = len(token_ids)
  if verbose: print(f'The text has been split into {num_tokens} tokens')
  # Compute proportion of continued words
  continued_words = float(compute_continued_words(token_ids, tokenizer, tokenizer.all_special_tokens, text))
  if verbose: print(f'The number of continued words in the text is {continued_words}')
  return (num_tokens, continued_words)

In [16]:
def run_experiment(tokenizer_id, hf_access_token, data_df, sample_prop, verbose=False):
  # Load tokenizer
  tokenizer = AutoTokenizer.from_pretrained(tokenizer_id, token=hf_access_token)

  # Calculate sample
  s_data_df = data_df.sample(frac=sample_prop, axis=0)
  num_docs = s_data_df.shape[0]

  # Create tokenizer obj
  tokenizer_obj = {
    'tokenizer': tokenizer_id,
    'vocab_size': tokenizer.vocab_size,
    'num_docs': num_docs,
    'num_words': [],
    'num_tokens': [],
    'fertility': [],
    'num_continued_words': [],
    'prop_continued_words': []
  }

  # Iterate over text
  for i in tqdm(range(num_docs), desc=f'Runnning experiment on {tokenizer_id}'):
    text = data_df.loc[i, 'text']
    num_words = int(data_df.loc[i, "num_words_punct_spacy"])
    if verbose: print(f'The text has {num_words} words')
    num_tokens, num_continued_words = process_text(text, tokenizer, verbose)
    fertility = compute_fertility(num_words, num_tokens)
    # Add to dictionary
    tokenizer_obj['num_words'].append(num_words)
    tokenizer_obj['num_tokens'].append(num_tokens)
    tokenizer_obj['fertility'].append(fertility)
    tokenizer_obj['num_continued_words'].append(num_continued_words)
    tokenizer_obj['prop_continued_words'].append(num_continued_words/num_words)

  return tokenizer_obj

In [17]:
def compute_average_metrics(tokenizer_obj):
  # Compute average metrics
  tokenizer_obj['avg_num_words'] = float(sum(tokenizer_obj['num_words'])/tokenizer_obj['num_docs'])
  tokenizer_obj['avg_num_tokens'] = float(sum(tokenizer_obj['num_tokens'])/tokenizer_obj['num_docs'])
  return tokenizer_obj

In [18]:
def save_output_to_json(tokenizers_exp):
  output_file_path = os.path.join(project_dir, 'tokenizers_experiment_all.json')
  with open(output_file_path, 'w', encoding='utf-8') as output_file:
    json.dump(tokenizers_exp, output_file, indent=4, ensure_ascii=False)

### Run experiments

In [19]:
tokenizers_exp = {}

In [20]:
tokenizer_ids = [
    'google/gemma-3-1b-it',
    'google/gemma-3-4b-it',
    'Qwen/Qwen3-32B',
    'Xenova/gpt-4o',
    'meta-llama/Llama-3.1-8B',
    'microsoft/phi-4',
    'CohereForAI/aya-101',
    'google/madlad400-3b-mt',
    'MaLA-LM/mala-500-10b-v2',
    'MaLA-LM/emma-500-llama3.1-8b-bi',
    'rubuntu/gemma-2-9b-it-SimPO-Jopara-V3.4',
    'mmaguero/multilingual-bert-gn-base-cased',
    'mmaguero/gn-bert-tiny-cased'
]

In [21]:
#for tokenizer_id in tokenizer_ids:
#  tokenizer = AutoTokenizer.from_pretrained(tokenizer_id, token=hf_access_token)
#  print(f'Tokenizer: {tokenizer.name_or_path}')
#  text = data_df.loc[0, 'text']
#  print(f'Text: {text}')
#  num_words = int(data_df.loc[0, "num_words_punct_spacy"])
#  print(f'The text has {num_words} words')
#  num_tokens, num_continued_words = process_text(text, tokenizer, verbose=True)
#  print('\n\n')

In [22]:
if OVERWRITE_EXP or not os.path.exists(os.path.join(project_dir, 'tokenizers_experiment_all.json')):
  for tokenizer_id in tokenizer_ids:
    tokenizer_name = tokenizer_id.split('/')[1].split('-')[0].lower()
    tokenizers_exp[tokenizer_name] = run_experiment(tokenizer_id, hf_access_token, data_df, sample_prop=1)
    tokenizers_exp[tokenizer_name] = compute_average_metrics(tokenizers_exp[tokenizer_name])
    print('Saving current results to json')
    save_output_to_json(tokenizers_exp)

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Runnning experiment on mmaguero/gn-bert-tiny-cased: 100%|██████████| 13371/13371 [02:05<00:00, 106.29it/s]


Saving current results to json


## Analyze results

In [23]:
with open(os.path.join(project_dir, 'tokenizers_experiment_all.json'), 'r', encoding='utf-8') as input_file:
  tokenizers_exp = json.load(input_file)

### Subword Fertility

Subword fertility measures how many tokens are generated by word on average. For each document, we divide the number of tokens in which the document is split by number of words of the document. Then, results by each document are averaged.

In [31]:
tokenizers = list(tokenizers_exp.keys())
fertility = []
for tokenizer in tokenizers:
  name = tokenizers_exp[tokenizer]['tokenizer'].split('/')[1].title()
  vocab_size = tokenizers_exp[tokenizer]['vocab_size']
  fert = sum(tokenizers_exp[tokenizer]['num_tokens'])/sum(tokenizers_exp[tokenizer]['num_words'])
  avg_prop_continued_words = sum(tokenizers_exp[tokenizer]['prop_continued_words'])/len(tokenizers_exp[tokenizer]['prop_continued_words'])
  fertility.append(
      {
          'tokenizer': f'{name}<br>(V={vocab_size})',
          'fertility_score': fert,
          'fertility_txt': round(fert, 3),
          'total_words': sum(tokenizers_exp[tokenizer]['num_words']),
          'total_tokens': sum(tokenizers_exp[tokenizer]['num_tokens']),
          'avg_words': tokenizers_exp[tokenizer]['avg_num_words'],
          'avg_tokens': tokenizers_exp[tokenizer]['avg_num_tokens'],
          'avg_tokens_txt': round(tokenizers_exp[tokenizer]['avg_num_tokens'], 3),
          'avg_prop_continued_words': avg_prop_continued_words,
          'avg_prop_continued_words_txt': round(avg_prop_continued_words, 3)
      }
  )
fertility_df = pd.DataFrame(fertility)
fertility_df

,tokenizer,fertility_score,fertility_txt,total_words,total_tokens,avg_words,avg_tokens,avg_tokens_txt,avg_prop_continued_words,avg_prop_continued_words_txt
0,Gemma-2-9B-It-Simpo-Jopara-V3.4<br>(V=256000),2.079155,2.079,7543822,15684778,564.192805,1173.044499,1173.044,0.871013,0.871
1,Qwen3-32B<br>(V=151643),2.310279,2.310,7543822,17428334,564.192805,1303.442824,1303.443,0.995264,0.995
2,Gpt-4O<br>(V=200000),1.950903,1.951,7543822,14717266,564.192805,1100.685513,1100.686,0.793341,0.793
3,Llama-3.1-8B<br>(V=128000),2.285466,2.285,7543822,17241147,564.192805,1289.443348,1289.443,0.983890,0.984
4,Phi-4<br>(V=100352),2.347734,2.348,7543822,17710891,564.192805,1324.574901,1324.575,1.017659,1.018
5,Aya-101<br>(V=250100),2.172551,2.173,7543822,16389338,564.192805,1225.737641,1225.738,1.065923,1.066
6,Madlad400-3B-Mt<br>(V=256000),1.810328,1.810,7543822,13656792,564.192805,1021.374018,1021.374,0.459768,0.460
7,Mala-500-10B-V2<br>(V=260164),1.814134,1.814,7543822,13685506,564.192805,1023.521502,1023.522,0.940432,0.940
8,Emma-500-Llama3.1-8B-Bi<br>(V=128000),2.285466,2.285,7543822,17241147,564.192805,1289.443348,1289.443,0.983890,0.984
9,Multilingual-Bert-Gn-Base-Cased<br>(V=119547),2.152623,2.153,7543822,16239006,564.192805,1214.494503,1214.495,1.122681,1.123


In [35]:
fertility_df.sort_values(by='fertility_score', ascending=True, inplace=True)
fertility_df.reset_index(drop=True, inplace=True)
color_palette=px.colors.qualitative.Vivid
fig = px.bar(fertility_df, x='fertility_score', y='tokenizer', template='none',
             color='tokenizer', text='fertility_txt',
             color_discrete_sequence=color_palette)
fig.update_layout(autosize=True, xaxis_title='Subword Fertility', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=11)), width=1200,
                  margin=dict(l=200))
fig.show()

In [50]:
color_palette = px.colors.qualitative.Vivid
fig = go.Figure()

fig.add_trace(go.Bar(
    x=fertility_df['tokenizer'],
    y=fertility_df['avg_tokens'],
    name='Tokens',
    marker_color=color_palette,
    text=fertility_df['avg_tokens_txt'],
))

fig.add_trace(go.Scatter(
    x=fertility_df['tokenizer'],
    y=fertility_df['fertility_score'],
    name='Fertility Score',
    mode='markers',
    yaxis='y2',
    text=fertility_df['fertility_txt'],
    textposition='top center',
    marker=dict(
      color='gray',
      size=10,
      line=dict(
        color='black',
        width=2
      )
  )
))

# Add a horizontal dashed line at the average number of words
avg_words = fertility_df['avg_words'].mean()
fig.add_shape(type='line',
              x0=-0.5, y0=avg_words,
              x1=len(fertility_df['tokenizer'])-0.5, y1=avg_words,
              line=dict(color='black', width=2, dash='dash'))

# Add text annotation for the average number of words
fig.add_annotation(
    x=len(fertility_df['tokenizer'])-2.5,
    y=avg_words,
    text=f'Average Words per Document: {avg_words:.2f}',
    showarrow=False,
    yshift=10,
    font=dict(
        color='black'
    )
)

fig.update_layout(
    width=1000,
    height=900,
    yaxis=dict(
        title='Average tokens per document'
    ),
    xaxis=dict(
        tickangle=90,
    ),
    yaxis2=dict(
        title='Subword fertility',
        overlaying='y',
        side='right'
    ),
    template='none',
    legend=dict(
        x=1.1,
        y=1
    )
)
fig.show()

## Proportion of continued words

This metric measures the tendency of a tokenizer to split up words. For each document, it is calculated by dividing the number of split words by number of words of the document. Then, results by each document are averaged.

In [53]:
color_palette=px.colors.qualitative.Vivid
fertility_df.sort_values(by='avg_prop_continued_words', ascending=True, inplace=True)
fig = px.bar(fertility_df, x='avg_prop_continued_words', y='tokenizer', template='none',
             color='tokenizer', text='avg_prop_continued_words_txt',
             color_discrete_sequence=color_palette)
fig.update_layout(autosize=True, xaxis_title='Average proportion of continued words per document', yaxis_title='',
                  showlegend=False, yaxis=dict(tickfont=dict(size=11)), width=1200,
                  margin=dict(l=200))
fig.show()